# GCP Dataset — Exploratory Data Analysis
Deep dive into the training data before designing the pipeline.

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

sys.path.insert(0, '..')
from src.dataset import load_labels

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('../train_dataset')          # adjust to your path
LABELS_JSON = DATA_ROOT / 'curated_gcp_marks.json'

# load_labels applies shape-string normalization (e.g. 'L-Shape' -> 'L-Shaped')
# and fills in the 4 manually-reviewed missing verified_shape entries.
labels = load_labels(str(LABELS_JSON))

print(f'Total annotated images (after cleaning): {len(labels)}')

## 1. Shape class distribution

In [ ]:
shapes = [v['verified_shape'] for v in labels.values()]
counts = Counter(shapes)

fig, ax = plt.subplots(figsize=(7, 4))
classes = sorted(counts.keys())
vals = [counts[c] for c in classes]
bars = ax.bar(classes, vals, color=['#5DCAA5', '#EF9F27', '#7F77DD'])
ax.bar_label(bars, labels=[f'{v}\n({100*v/sum(vals):.1f}%)' for v in vals], padding=3)
ax.set_ylabel('Count')
ax.set_title('Shape class distribution')
ax.set_ylim(0, max(vals) * 1.3)
plt.tight_layout()
plt.savefig('eda_shape_dist.png', dpi=120)
plt.show()

# Imbalance ratio
min_cls = min(vals)
max_cls = max(vals)
print(f'Imbalance ratio (max/min): {max_cls/min_cls:.1f}x')
print('→ Weighted sampler and focal loss recommended')

## 2. Keypoint coordinate distribution

In [ ]:
xs = np.array([v['mark']['x'] for v in labels.values()])
ys = np.array([v['mark']['y'] for v in labels.values()])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# X distribution
axes[0].hist(xs, bins=40, color='#378ADD', alpha=0.8)
axes[0].set_title('X coordinate distribution')
axes[0].set_xlabel('x (pixels)')

# Y distribution
axes[1].hist(ys, bins=40, color='#1D9E75', alpha=0.8)
axes[1].set_title('Y coordinate distribution')
axes[1].set_xlabel('y (pixels)')

# 2D scatter
axes[2].scatter(xs, ys, alpha=0.3, s=10, c='#534AB7')
axes[2].set_title('Keypoint 2D scatter')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
axes[2].invert_yaxis()  # image coords: y increases downward

plt.tight_layout()
plt.savefig('eda_coords.png', dpi=120)
plt.show()

print(f'X: mean={xs.mean():.0f}, std={xs.std():.0f}, min={xs.min():.0f}, max={xs.max():.0f}')
print(f'Y: mean={ys.mean():.0f}, std={ys.std():.0f}, min={ys.min():.0f}, max={ys.max():.0f}')

## 3. Image resolution check

In [ ]:
resolutions = []
missing = []
SAMPLE_LIMIT = 200  # don't open all images; sample

sampled_paths = list(labels.keys())[:SAMPLE_LIMIT]
for rel_path in sampled_paths:
    img_path = DATA_ROOT / rel_path
    if not img_path.exists():
        missing.append(rel_path)
        continue
    with Image.open(img_path) as img:
        resolutions.append(img.size)  # (W, H)

unique = Counter(resolutions)
print(f'Unique resolutions in first {SAMPLE_LIMIT} images: {len(unique)}')
for res, cnt in unique.most_common():
    print(f'  {res[0]}x{res[1]}: {cnt}')
print(f'Missing files: {len(missing)}')

## 4. Sample image visualization with GCP annotation

In [ ]:
import random
random.seed(42)

# Show 2 examples per shape class
by_class = {'Cross': [], 'L-Shaped': [], 'Square': []}
for path, ann in labels.items():
    by_class[ann['verified_shape']].append((path, ann))

fig, axes = plt.subplots(3, 2, figsize=(14, 18))
colors = {'Cross': '#EF9F27', 'L-Shaped': '#D4537E', 'Square': '#378ADD'}

for row_idx, (cls_name, samples) in enumerate(by_class.items()):
    chosen = random.sample(samples, min(2, len(samples)))
    for col_idx, (rel_path, ann) in enumerate(chosen):
        img_path = DATA_ROOT / rel_path
        if not img_path.exists():
            continue
        ax = axes[row_idx][col_idx]
        img = Image.open(img_path)
        W, H = img.size
        x, y = ann['mark']['x'], ann['mark']['y']
        # Crop a 400px window around the GCP for visibility
        half = 200
        left = max(0, int(x) - half)
        top  = max(0, int(y) - half)
        right  = min(W, int(x) + half)
        bottom = min(H, int(y) + half)
        crop = img.crop((left, top, right, bottom))
        ax.imshow(crop)
        # Draw crosshair
        cx = x - left
        cy = y - top
        c = colors[cls_name]
        ax.axhline(cy, color=c, linewidth=1, alpha=0.8)
        ax.axvline(cx, color=c, linewidth=1, alpha=0.8)
        ax.plot(cx, cy, 'o', color=c, markersize=12, markeredgecolor='white', markeredgewidth=2)
        ax.set_title(f'{cls_name}  |  {Path(rel_path).name}', fontsize=10)
        ax.axis('off')

plt.suptitle('Sample GCP annotations (400×400 crop around marker)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('eda_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Near-edge keypoint analysis

In [ ]:
# Flag keypoints that are very close to image borders
EDGE_THRESH = 50
IMG_W, IMG_H = 2048, 1365  # expected resolution

near_edge = []
for path, ann in labels.items():
    x, y = ann['mark']['x'], ann['mark']['y']
    if x < EDGE_THRESH or x > IMG_W - EDGE_THRESH or y < EDGE_THRESH or y > IMG_H - EDGE_THRESH:
        near_edge.append((path, x, y))

print(f'Keypoints within {EDGE_THRESH}px of image border: {len(near_edge)} / {len(labels)}')
print('Mitigation: pad crop region and clip coordinates during augmentation')
for p, x, y in near_edge[:10]:
    print(f'  x={x:.0f}, y={y:.0f}  →  {p}')

## 6. Per-project statistics

In [ ]:
project_stats = {}
for rel_path, ann in labels.items():
    proj = Path(rel_path).parts[0]
    if proj not in project_stats:
        project_stats[proj] = Counter()
    project_stats[proj][ann['verified_shape']] += 1

print(f'Projects: {len(project_stats)}')
for proj, shape_dist in sorted(project_stats.items(), key=lambda x: -sum(x[1].values())):
    total = sum(shape_dist.values())
    dist_str = ', '.join(f'{k}:{v}' for k,v in sorted(shape_dist.items()))
    print(f'  {proj:30s}: {total:4d} images  [{dist_str}]')